##dosya erişimi

In [ ]:
import os
PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"

print("BitirmeProjesi klasör yapısı:")
for root, dirs, files in os.walk(PROJECT_FOLDER):
    # __pycache__ ve .git gibi gizli klasörleri atla
    dirs[:] = [d for d in dirs if not d.startswith('.') and d != '__pycache__']
    level = root.replace(PROJECT_FOLDER, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root) or 'BitirmeProjesi'}/")
    sub_indent = '  ' * (level + 1)
    for f in sorted(files):
        size_kb = os.path.getsize(os.path.join(root, f)) / 1024
        print(f"{sub_indent}{f} ({size_kb:.0f} KB)")

BitirmeProjesi klasör yapısı:
BitirmeProjesi/
  LlmSorular.2.csv (202 KB)
  LlmSorular.csv (13 KB)
  LlmSorular.gsheet (0 KB)
  Sorular e-tablo - Base_Questions 489.csv (511 KB)
  Sorular e-tablo - Variants (1).csv (258 KB)
  acik_kaynak_analiz.ipynb (850 KB)
  sample_questions_table.csv (5 KB)
  sample_questions_table.png (325 KB)
  checkpoints/
    generation_20260306_1228.json (342 KB)
    generation_20260306_1437.json (332 KB)
    generation_20260307_1119.json (423 KB)
    generation_20260308_1206.json (9 KB)
    generation_20260321_1222.json (186 KB)
    generation_20260321_1351.json (524 KB)
    generation_llama31_8b_base_20260514.json (785 KB)
    generation_llama31_8b_base_20260514_backup_20260514_195745.json (672 KB)
    generation_llama31_8b_varyant_20260517.json (376 KB)
    generation_qwen25_7b_20260513.csv (1025 KB)
    generation_qwen25_7b_base_20260513.json (623 KB)
    generation_qwen25_7b_varyant_20260515.json (330 KB)
    generation_trendyol_7b_base_20260514.json (140

In [ ]:
# Büyük modellerin CSV yapısını ve varyant dosyasını incele
import pandas as pd
import os

PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
RESULTS = f"{PROJECT_FOLDER}/results/"

files_to_inspect = [
    "evaluation_soft_20260306_1325.csv",        # GPT-4o-mini base
    "evaluation_soft_20260306_1515gpt4.csv",    # GPT-4o base
    "evaluation_soft_20260308_1141.csv",        # Claude Sonnet 4 base
    "evaluation_soft_20260321_1211.csv",        # Gemini 2.5 Flash base
    "variant_evaluation_20260414_1440.csv",     # 3 büyük modelin varyantı
    "variant_comparison_20260414_1445.csv",     # 3 modelin karşılaştırması
]

for f in files_to_inspect:
    path = os.path.join(RESULTS, f)
    if not os.path.exists(path):
        print(f"\n✗ YOK: {f}")
        continue

    df = pd.read_csv(path)
    print(f"\n{'='*70}")
    print(f"DOSYA: {f}")
    print(f"  Satır: {len(df)}, Kolon: {len(df.columns)}")
    print(f"  Kolonlar: {list(df.columns)}")

    # Modeli tahmin et: model_name veya benzeri bir kolon var mı?
    model_cols = [c for c in df.columns if 'model' in c.lower() and 'name' in c.lower()]
    if model_cols:
        print(f"  Model kolonu var: {model_cols[0]} → {df[model_cols[0]].unique()[:5]}")

    # İlk satır örneği
    print(f"\n  İlk satır:")
    for col in df.columns[:8]:
        val = str(df.iloc[0][col])[:80]
        print(f"     {col}: {val}")


DOSYA: evaluation_soft_20260306_1325.csv
  Satır: 488, Kolon: 31
  Kolonlar: ['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count', 'model_reasoning', 'model_answer', 'model_confidence', 'error', 'semantic_similarity', 'nli_score', 'faithfulness_score', 'evidence_recall', 'exact_match', 'accuracy_score', 'confidence_normalized', 'epistemic_gap', 'hallucination_risk', 'calibration_type', 'reasoning_length', 'evidence_length', 'length_ratio', 'soft_match', 'soft_accuracy_score', 'match_type', 'answer_semantic_sim', 'answer_nli_entailment']

  İlk satır:
     question_id: NR-1-001
     category: numerical_reasoning
     difficulty: easy
     context: Sevda C sayısı ile D sayısını toplamak istiyor. Ancak, toplama işlemini yapmadan
     question: Buna göre, D sayısı kaçtır?
     gold_evidence: Sevda C sayısı ile D sayısını toplamak istiyor. | Sonucun C sayısının yüzler bas
     gold_answer: 400
     gold_evid

In [ ]:
# ============================================================
# BU KODU COLAB'DA HERHANGİ BİR NOTEBOOK'TA ÇALIŞTIR
# Drive bağlı olması yeterli
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import os
import json
import pandas as pd

PROJECT = "/content/drive/MyDrive/BitirmeProjesi"

print("=== CHECKPOINT KLASÖRÜ ===")
cp_path = f"{PROJECT}/checkpoints/"
if os.path.exists(cp_path):
    for f in sorted(os.listdir(cp_path)):
        fpath = os.path.join(cp_path, f)
        size = os.path.getsize(fpath)
        print(f"  {f} ({size/1024:.0f} KB)")

        # JSON ise ilk kaydı göster
        if f.endswith('.json'):
            try:
                with open(fpath, 'r', encoding='utf-8') as jf:
                    data = json.load(jf)
                if isinstance(data, list) and len(data) > 0:
                    print(f"    İlk kayıt keys: {list(data[0].keys())}")
                    print(f"    Toplam kayıt: {len(data)}")
                    if 'model_reasoning' in data[0] and data[0]['model_reasoning']:
                        print(f"    İlk reasoning: {str(data[0]['model_reasoning'])[:200]}...")
            except:
                pass

        # CSV ise sütunları göster
        if f.endswith('.csv'):
            try:
                df = pd.read_csv(fpath, nrows=2)
                print(f"    Sütunlar: {list(df.columns)}")
                print(f"    Satır sayısı: {len(pd.read_csv(fpath))}")
                if 'model_reasoning' in df.columns and df['model_reasoning'].notna().any():
                    print(f"    İlk reasoning: {str(df['model_reasoning'].iloc[0])[:200]}...")
            except:
                pass
        print()

print("\n=== RESULTS KLASÖRÜ ===")
res_path = f"{PROJECT}/results/"
if os.path.exists(res_path):
    for f in sorted(os.listdir(res_path)):
        fpath = os.path.join(res_path, f)
        size = os.path.getsize(fpath)
        print(f"  {f} ({size/1024:.0f} KB)")

        if f.endswith('.csv'):
            try:
                df_full = pd.read_csv(fpath)
                print(f"    Sütunlar: {list(df_full.columns)}")
                print(f"    Satır sayısı: {len(df_full)}")
                if 'model_reasoning' in df_full.columns and df_full['model_reasoning'].notna().any():
                    print(f"    İlk reasoning: {str(df_full['model_reasoning'].iloc[0])[:200]}...")
                if 'faithfulness_score' in df_full.columns:
                    print(f"    Ort. faithfulness: {df_full['faithfulness_score'].mean():.3f}")
                    print(f"    Ort. accuracy: {df_full['accuracy_score'].mean():.3f}")
            except:
                pass
        print()
else:
    print("  Results klasörü bulunamadı!")

Mounted at /content/drive
=== CHECKPOINT KLASÖRÜ ===
  generation_20260306_1228.json (342 KB)
    İlk kayıt keys: ['question_id', 'model_reasoning', 'model_answer', 'model_confidence', 'error']
    Toplam kayıt: 488
    İlk reasoning: Sevda, C sayısı ile D sayısını toplamak istiyor. Ancak, toplama işlemi yapmadan, sonucun C sayısının yüzler basamağını 4 artırarak elde edilebileceğini fark ediyor. Bu durumda, D sayısı, C sayısının y...

  generation_20260306_1437.json (332 KB)
    İlk kayıt keys: ['question_id', 'model_reasoning', 'model_answer', 'model_confidence', 'error']
    Toplam kayıt: 488
    İlk reasoning: Sevda, C sayısı ile D sayısını toplamak istiyor. Ancak, sonucun C sayısının yüzler basamağını 4 artırarak elde edilebileceğini fark ediyor. Bu, D sayısının C sayısının yüzler basamağını 4 artıracak bi...

  generation_20260307_1119.json (423 KB)
    İlk kayıt keys: ['question_id', 'model_reasoning', 'model_answer', 'model_confidence', 'error']
    Toplam kayıt: 488
    İlk re

In [ ]:
import os
from datetime import datetime

PROJECT = "/content/drive/MyDrive/BitirmeProjesi"

print("=== TÜM DOSYALAR ZAMANLA SIRALI ===\n")

all_files = []

for folder in ['checkpoints', 'results']:
    folder_path = f"{PROJECT}/{folder}/"
    if os.path.exists(folder_path):
        for f in os.listdir(folder_path):
            fpath = os.path.join(folder_path, f)
            mtime = os.path.getmtime(fpath)
            dt = datetime.fromtimestamp(mtime)
            size = os.path.getsize(fpath)
            all_files.append((dt, folder, f, size))

# Zamana göre sırala
all_files.sort(key=lambda x: x[0])

for dt, folder, fname, size in all_files:
    print(f"  {dt.strftime('%Y-%m-%d %H:%M:%S')} | {folder:12s} | {fname} ({size/1024:.0f} KB)")

=== TÜM DOSYALAR ZAMANLA SIRALI ===

  2026-03-06 13:03:43 | checkpoints  | results_generation_20260306_1228.csv (941 KB)
  2026-03-06 13:03:53 | checkpoints  | generation_20260306_1228.json (342 KB)
  2026-03-06 13:08:19 | results      | evaluation_20260306_1308.csv (990 KB)
  2026-03-06 13:08:45 | results      | evaluation_soft_20260306_1308.csv (1006 KB)
  2026-03-06 13:25:51 | results      | evaluation_soft_20260306_1325.csv (1006 KB)
  2026-03-06 13:26:13 | results      | quadrant_gpt4omini.png (193 KB)
  2026-03-06 13:30:26 | results      | hallucination_gpt4omini.png (65 KB)
  2026-03-06 13:30:29 | results      | exact_vs_soft_gpt4omini.png (92 KB)
  2026-03-06 13:30:32 | results      | category_difficulty_gpt4omini.png (74 KB)
  2026-03-06 15:10:45 | checkpoints  | results_generation_20260306_1437.csv (930 KB)
  2026-03-06 15:10:56 | checkpoints  | generation_20260306_1437.json (332 KB)
  2026-03-06 15:14:45 | results      | evaluation_20260306_1514.csv (974 KB)
  2026-03-06 15

In [ ]:
from google.colab import userdata

# API key kontrolü
keys = {}
for key_name in ['OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GOOGLE_API_KEY']:
    try:
        val = userdata.get(key_name)
        keys[key_name] = "✓ VAR" if val else "✗ YOK"
    except:
        keys[key_name] = "✗ YOK"

for k, v in keys.items():
    print(f"  {k}: {v}")

  OPENAI_API_KEY: ✓ VAR
  ANTHROPIC_API_KEY: ✓ VAR
  GOOGLE_API_KEY: ✓ VAR


#varyant

In [ ]:
# VARYANT ANALİZİ - BÖLÜM 1: CONFIG & SETUP

# Drive bağla
from google.colab import drive
drive.mount('/content/drive')
!pip install anthropic google-generativeai -q

import sys
import os
sys.path.insert(0, '/content/drive/MyDrive/BitirmeProjesi')

# Kontrol
print("Path eklendi:", '/content/drive/MyDrive/BitirmeProjesi' in sys.path)
utils_path = '/content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py'
print("Utils dosyası mevcut:", os.path.exists(utils_path))

from core.faithfulness_utils import *

from google.colab import userdata
from openai import OpenAI
from anthropic import Anthropic
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import pandas as pd
import numpy as np
import json
from datetime import datetime

# --- Path Ayarları ---
PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"

# Varyant verisi - bu dosyayı Drive'a yüklemen lazım
VARIANT_DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Variants (1).csv"

# Base evaluation dosyaları (soft accuracy)
BASE_EVAL_FILES = {
    "gpt-4o-mini": f"{RESULTS_PATH}evaluation_soft_20260306_1308gpt4omini.csv",
    "gpt-4o": f"{RESULTS_PATH}evaluation_soft_20260306_1515gpt4.csv",
    "claude-sonnet-4-20250514": f"{RESULTS_PATH}evaluation_soft_20260308_1141.csv",
    "gemini-2.5-flash": f"{RESULTS_PATH}evaluation_soft_20260321_1211.csv"

}

# --- API Keys ---
OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    GOOGLE_API_KEY = None

# --- Model Ayarları ---
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"
NLI_MODEL = "joeddav/xlm-roberta-large-xnli"

# 4 model listesi
MODEL_LIST = ["gpt-4o-mini", "gpt-4o", "claude-sonnet-4-20250514", "gemini-2.5-flash"]

# --- Doğrulama ---
print("\n✓ Config yüklendi")
print(f"  Proje: {PROJECT_FOLDER}")
print(f"  OpenAI API: {'✓' if OPENAI_API_KEY else '✗'}")
print(f"  Anthropic API: {'✓' if ANTHROPIC_API_KEY else '✗'}")
print(f"  Google API: {'✓' if GOOGLE_API_KEY else '✗'}")

# Dosya kontrolü
print("\n► Dosya kontrolleri:")
print(f"  Varyant CSV: {'✓' if os.path.exists(VARIANT_DATA_PATH) else '✗ BULUNAMADI!'}")
for model, path in BASE_EVAL_FILES.items():
    exists = os.path.exists(path)
    print(f"  {model} base eval: {'✓' if exists else '✗ BULUNAMADI!'}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Path eklendi: True
Utils dosyası mevcut: True

✓ Config yüklendi
  Proje: /content/drive/MyDrive/BitirmeProjesi
  OpenAI API: ✓
  Anthropic API: ✓
  Google API: ✓

► Dosya kontrolleri:
  Varyant CSV: ✓
  gpt-4o-mini base eval: ✓
  gpt-4o base eval: ✓
  claude-sonnet-4-20250514 base eval: ✓
  gemini-2.5-flash base eval: ✓


In [ ]:
# VARYANT ANALİZİ - BÖLÜM 2: MODELLERİ YÜKLE

print("► Modeller yükleniyor...")

# GPU kontrolü
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Cihaz: {device.upper()}")

# OpenAI client
openai_client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else None
print(f"  OpenAI client: {'✓' if openai_client else '✗'}")

# Claude client
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
print(f"  Claude client: {'✓' if claude_client else '✗'}")

# Gemini client
gemini_model_client = None
if GOOGLE_API_KEY:
    genai.configure(api_key=GOOGLE_API_KEY)
    gemini_model_client = genai.GenerativeModel('gemini-2.5-flash')
print(f"  Gemini client: {'✓' if gemini_model_client else '✗'}")

# Embedding model
print("  ► Embedding model yükleniyor...")
emb_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"  ✓ {EMBEDDING_MODEL} yüklendi")

# NLI model
print("  ► NLI model yükleniyor...")
nli_pipe = pipeline(
    "zero-shot-classification",
    model=NLI_MODEL,
    device=0 if device == "cuda" else -1
)
print(f"  ✓ {NLI_MODEL} yüklendi")

print("\n✓ Tüm modeller hazır!")

► Modeller yükleniyor...
  Cihaz: CUDA
  OpenAI client: ✓
  Claude client: ✓
  Gemini client: ✓
  ► Embedding model yükleniyor...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

  ✓ joeddav/xlm-roberta-large-xnli yüklendi

✓ Tüm modeller hazır!


In [ ]:
# VARYANT ANALİZİ - BÖLÜM 3: VERİLERİ YÜKLE VE DOĞRULA
# --- 1. Varyant verisini yükle ---
df_variants = pd.read_csv(VARIANT_DATA_PATH)
print(f"► Varyant verisi yüklendi: {len(df_variants)} satır")

# Eksik veri kontrolü
initial_count = len(df_variants)
df_variants = df_variants.dropna(subset=['context', 'question', 'gold_answer', 'gold_evidence'])
dropped = initial_count - len(df_variants)
if dropped > 0:
    print(f"  ⚠️ {dropped} satır eksik veri nedeniyle çıkarıldı")
print(f"  Temiz varyant sayısı: {len(df_variants)}")

# Gold evidence'ı listeye çevir (base pipeline ile aynı)
df_variants['gold_evidence_list'] = df_variants['gold_evidence'].apply(
    lambda x: [item.strip() for item in str(x).split('|')]
)

print(f"\n  Kategoriler: {sorted(df_variants['category'].unique().tolist())}")
print(f"  Zorluk seviyeleri: {sorted(df_variants['difficulty'].unique().tolist())}")
print(f"  Varyant tipleri: {sorted(df_variants['variant_type'].dropna().unique().tolist())}")
print(f"  Unique base soru sayısı: {df_variants['base_question_id'].nunique()}")

# Kategori başına dağılım
print(f"\n  Kategori dağılımı:")
for cat, count in df_variants['category'].value_counts().items():
    print(f"    {cat}: {count}")

# --- 2. Base evaluation dosyalarını yükle ---
print("\n► Base evaluation dosyaları yükleniyor...")
base_evals = {}

for model_name, path in BASE_EVAL_FILES.items():
    df_base = pd.read_csv(path)
    base_evals[model_name] = df_base
    print(f"  {model_name}: {len(df_base)} satır, "
          f"ort. faithfulness={df_base['faithfulness_score'].mean():.3f}, "
          f"ort. soft_accuracy={df_base['soft_accuracy_score'].mean():.3f}")

# --- 3. Eşleştirme kontrolü ---
print("\n► Base-Varyant eşleştirme kontrolü:")
variant_bases = set(df_variants['base_question_id'].str.strip().unique())

for model_name, df_base in base_evals.items():
    base_ids = set(df_base['question_id'].str.strip().unique())
    matched = variant_bases.intersection(base_ids)
    missing = variant_bases - base_ids
    print(f"  {model_name}: {len(matched)}/{len(variant_bases)} base eşleşti", end="")
    if missing:
        print(f" (eksik: {len(missing)} soru)")
    else:
        print(" ✓")

print("\n✓ Veri yükleme ve doğrulama tamamlandı!")

► Varyant verisi yüklendi: 257 satır
  Temiz varyant sayısı: 257

  Kategoriler: ['analogical_reasoning', 'causal_reasoning', 'compositional_reasoning', 'logical_inference', 'numerical_reasoning', 'temporal_reasoning']
  Zorluk seviyeleri: ['easy', 'hard', 'medium']
  Varyant tipleri: ['causal_marker', 'conditional_restructuring', 'evidentiality', 'negation_form', 'nominalization', 'paraphrase', 'passive_voice', 'quantifier_paraphrase', 'temporal_suffix', 'tense_change', 'word_order']
  Unique base soru sayısı: 85

  Kategori dağılımı:
    temporal_reasoning: 83
    causal_reasoning: 54
    logical_inference: 33
    numerical_reasoning: 30
    analogical_reasoning: 30
    compositional_reasoning: 27

► Base evaluation dosyaları yükleniyor...
  gpt-4o-mini: 488 satır, ort. faithfulness=0.676, ort. soft_accuracy=0.598
  gpt-4o: 488 satır, ort. faithfulness=0.657, ort. soft_accuracy=0.707
  claude-sonnet-4-20250514: 488 satır, ort. faithfulness=0.667, ort. soft_accuracy=0.629
  gemini-2.5

#test

In [ ]:
# VARYANT ANALİZİ - BÖLÜM 4: GENERATION (4 MODEL)

SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""


def generate_single_variant(context, question, model_name):
    """Tek bir varyant için belirtilen modelden cevap al"""

    user_prompt = f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki bağlamı kullanarak soruyu cevapla. Yanıtını JSON formatında ver."""

    try:
        content = None

        if model_name.startswith("claude"):
            response = claude_client.messages.create(
                model=model_name,
                max_tokens=2048,
                messages=[
                    {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_prompt}"}
                ]
            )
            content = response.content[0].text

        elif model_name.startswith("gemini"):
            full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
            response = gemini_model_client.generate_content(full_prompt)
            content = response.text

        else:  # OpenAI modelleri
            response = openai_client.chat.completions.create(
                model=model_name,
                temperature=0,
                max_tokens=2048,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content

        # JSON parse
        result = extract_json_from_response(content)

        if result is None:
            return {"error": "JSON parse edilemedi", "raw_response": content[:500]}

        for field in ['reasoning', 'answer', 'confidence']:
            if field not in result:
                result[field] = None

        return result

    except Exception as e:
        return {"error": str(e)}


def run_variant_generation(df_variants, model_list):
    """Tüm varyantları tüm modellerle çalıştır - checkpoint destekli"""
    from tqdm.notebook import tqdm

    checkpoint_file = f"{CHECKPOINT_PATH}variant_generation_all_models.json"
    os.makedirs(CHECKPOINT_PATH, exist_ok=True)

    # Mevcut checkpoint varsa yükle
    all_results = []
    processed_keys = set()

    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r', encoding='utf-8') as f:
            all_results = json.load(f)
            processed_keys = {(r['variant_id'], r['model']) for r in all_results}
            print(f"✓ Checkpoint yüklendi: {len(processed_keys)} sonuç mevcut")

    # Toplam iş hesapla
    total_work = len(df_variants) * len(model_list)
    remaining = total_work - len(processed_keys)
    print(f"► Toplam: {total_work}, Mevcut: {len(processed_keys)}, Kalan: {remaining}")

    if remaining == 0:
        print("✓ Tüm generation zaten tamamlanmış!")
        return pd.DataFrame(all_results)

    # Her model için her varyantı çalıştır
    for model_name in model_list:
        model_remaining = df_variants[
            ~df_variants['variant_id'].apply(lambda vid: (vid, model_name) in processed_keys)
        ]

        if len(model_remaining) == 0:
            print(f"  ✓ {model_name}: zaten tamamlanmış")
            continue

        print(f"\n► {model_name}: {len(model_remaining)} varyant işlenecek...")

        for idx, row in tqdm(model_remaining.iterrows(), total=len(model_remaining),
                             desc=f"{model_name}"):

            response = generate_single_variant(
                context=row['context'],
                question=row['question'],
                model_name=model_name
            )

            result_entry = {
                'variant_id': row['variant_id'],
                'base_question_id': row['base_question_id'],
                'category': row['category'],
                'difficulty': row['difficulty'],
                'variant_type': row['variant_type'],
                'model': model_name,
                'model_reasoning': response.get('reasoning'),
                'model_answer': response.get('answer'),
                'model_confidence': response.get('confidence'),
                'error': response.get('error')
            }
            all_results.append(result_entry)
            processed_keys.add((row['variant_id'], model_name))

            # Her 10 sonuçta checkpoint kaydet
            if len(all_results) % 10 == 0:
                with open(checkpoint_file, 'w', encoding='utf-8') as f:
                    json.dump(all_results, f, ensure_ascii=False, indent=2)

            time.sleep(0.5)

        # Model bittiğinde checkpoint kaydet
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)
        print(f"  ✓ {model_name} tamamlandı ve kaydedildi")

    # Final kayıt
    with open(checkpoint_file, 'w', encoding='utf-8') as f:
        json.dump(all_results, f, ensure_ascii=False, indent=2)

    print(f"\n✓ Tüm generation tamamlandı: {len(all_results)} sonuç")
    return pd.DataFrame(all_results)


# --- ÇALIŞTIR ---
import time
print("=" * 60)
print("VARYANT GENERATION BAŞLIYOR")
print(f"  {len(df_variants)} varyant × {len(MODEL_LIST)} model = {len(df_variants) * len(MODEL_LIST)} API çağrısı")
print(f"  Tahmini süre: ~{len(df_variants) * len(MODEL_LIST) * 1.5 / 60:.0f} dakika")
print("=" * 60)

df_generation = run_variant_generation(df_variants, MODEL_LIST)

# Özet
print("\n► Generation Özeti:")
for model in MODEL_LIST:
    model_data = df_generation[df_generation['model'] == model]
    success = model_data['model_answer'].notna().sum()
    errors = model_data['error'].notna().sum()
    print(f"  {model}: {success} başarılı, {errors} hatalı")

# CSV olarak da kaydet
gen_csv_path = f"{CHECKPOINT_PATH}variant_generation_results.csv"
df_generation.to_csv(gen_csv_path, index=False, encoding='utf-8-sig')
print(f"\n✓ CSV kaydedildi: {gen_csv_path}")

VARYANT GENERATION BAŞLIYOR
  257 varyant × 4 model = 1028 API çağrısı
  Tahmini süre: ~26 dakika
► Toplam: 1028, Mevcut: 0, Kalan: 1028

► gpt-4o-mini: 257 varyant işlenecek...


gpt-4o-mini:   0%|          | 0/257 [00:00<?, ?it/s]

  ✓ gpt-4o-mini tamamlandı ve kaydedildi

► gpt-4o: 257 varyant işlenecek...


gpt-4o:   0%|          | 0/257 [00:00<?, ?it/s]

  ✓ gpt-4o tamamlandı ve kaydedildi

► claude-sonnet-4-20250514: 257 varyant işlenecek...


claude-sonnet-4-20250514:   0%|          | 0/257 [00:00<?, ?it/s]

  ✓ claude-sonnet-4-20250514 tamamlandı ve kaydedildi

► gemini-2.5-flash: 257 varyant işlenecek...


gemini-2.5-flash:   0%|          | 0/257 [00:00<?, ?it/s]

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1247.95ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1599.43ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 9615.67ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1575.53ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 633.72ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1015.35ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 6939.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding

  ✓ gemini-2.5-flash tamamlandı ve kaydedildi

✓ Tüm generation tamamlandı: 1028 sonuç

► Generation Özeti:
  gpt-4o-mini: 256 başarılı, 1 hatalı
  gpt-4o: 257 başarılı, 0 hatalı
  claude-sonnet-4-20250514: 249 başarılı, 8 hatalı
  gemini-2.5-flash: 8 başarılı, 249 hatalı

✓ CSV kaydedildi: /content/drive/MyDrive/BitirmeProjesi/checkpoints/variant_generation_results.csv


In [ ]:
# Sadece generate_single_variant fonksiyonunu tanımla, çalıştırma
import time

SYSTEM_PROMPT = """Sen bir problem çözme asistanısın. Sana verilen bağlam (context) ve soruyu (question) analiz edeceksin.

GÖREV:
1. Bağlamdaki bilgileri kullanarak adım adım düşün.
2. Her adımda hangi bilgiyi kullandığını açıkça belirt.
3. Sonuca ulaş ve cevabını ver.

ÇIKTI FORMATI (JSON):
{
    "reasoning": "Adım adım düşünce sürecin. Her adımda bağlamdan hangi bilgiyi kullandığını yaz.",
    "answer": "Nihai cevabın (kısa ve net)",
    "confidence": 0-100 arası bir sayı (cevabından ne kadar eminsin)
}

ÖNEMLİ:
- Sadece bağlamda verilen bilgileri kullan, dışarıdan bilgi ekleme.
- Cevabın SADECE geçerli bir JSON objesi olmalı, başka bir şey yazma.
- Türkçe yanıt ver."""


def generate_single_variant(context, question, model_name):
    user_prompt = f"""BAĞLAM:
{context}

SORU:
{question}

Yukarıdaki bağlamı kullanarak soruyu cevapla. Yanıtını JSON formatında ver."""

    try:
        content = None

        if model_name.startswith("claude"):
            response = claude_client.messages.create(
                model=model_name,
                max_tokens=2048,
                messages=[
                    {"role": "user", "content": f"{SYSTEM_PROMPT}\n\n{user_prompt}"}
                ]
            )
            content = response.content[0].text

        elif model_name.startswith("gemini"):
            full_prompt = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
            response = gemini_model_client.generate_content(full_prompt)
            content = response.text

        else:
            response = openai_client.chat.completions.create(
                model=model_name,
                temperature=0,
                max_tokens=2048,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}
                ],
                response_format={"type": "json_object"}
            )
            content = response.choices[0].message.content

        result = extract_json_from_response(content)

        if result is None:
            return {"error": "JSON parse edilemedi", "raw_response": content[:500]}

        for field in ['reasoning', 'answer', 'confidence']:
            if field not in result:
                result[field] = None

        return result

    except Exception as e:
        return {"error": str(e)}


print("✓ generate_single_variant fonksiyonu tanımlandı")

✓ generate_single_variant fonksiyonu tanımlandı


In [ ]:
# ============================================================
# GEMİNİ KOTA KONTROLÜ
# ============================================================
import time

# Test için bir varyant al
test_row = df_variants.iloc[0]

print(f"Test soru: {test_row['variant_id']}")
print("► Gemini API test ediliyor...")

try:
    response = generate_single_variant(
        context=test_row['context'],
        question=test_row['question'],
        model_name='gemini-2.5-flash'
    )

    if response.get('error') is None:
        print("✓ Gemini çalışıyor!")
        print(f"  Answer: {response.get('answer')}")
        print(f"  Confidence: {response.get('confidence')}")
    else:
        print(f"✗ Hata: {response.get('error')[:200]}")
except Exception as e:
    print(f"✗ Exception: {str(e)[:200]}")

Test soru: TR-01-017-v1
► Gemini API test ediliyor...
✓ Gemini çalışıyor!
  Answer: 13:00
  Confidence: 100


In [ ]:
# ============================================================
# GEMİNİ EKSİKLERİNİ TAMAMLA (DİKKATLİ MOD)
# ============================================================
import time
from tqdm.notebook import tqdm

checkpoint_file = f"{CHECKPOINT_PATH}variant_generation_all_models.json"

with open(checkpoint_file, 'r', encoding='utf-8') as f:
    all_results = json.load(f)

# Gemini hatalılarını bul
gemini_errors = [i for i, r in enumerate(all_results)
                 if r['model'] == 'gemini-2.5-flash' and r.get('error') is not None]

print(f"Gemini hatalı sonuç: {len(gemini_errors)}")
print("► Gemini tekrar çalıştırılıyor...")
print(f"  Strateji: 6 sn bekleme + 50'de bir 60 sn mola + rate limit retry")
print(f"  Tahmini süre: ~{(len(gemini_errors) * 6 + (len(gemini_errors) // 50) * 60) / 60:.0f} dakika\n")

fixed = 0
still_error = 0
rate_limit_waits = 0

for i, idx in enumerate(tqdm(gemini_errors, desc="Gemini retry")):
    r = all_results[idx]
    variant_id = r['variant_id']

    variant_row = df_variants[df_variants['variant_id'] == variant_id]
    if len(variant_row) == 0:
        still_error += 1
        continue
    variant_row = variant_row.iloc[0]

    # Her 50 istekte büyük mola
    if i > 0 and i % 50 == 0:
        print(f"\n  ⏸ 50 istek tamamlandı, 60 sn mola...")
        time.sleep(60)

    # Normal bekleme
    time.sleep(6)

    # Retry logic - rate limit hatası alırsa 2 kez daha dene
    max_retries = 3
    response = None
    for attempt in range(max_retries):
        try:
            response = generate_single_variant(
                context=variant_row['context'],
                question=variant_row['question'],
                model_name='gemini-2.5-flash'
            )

            # Rate limit hatası ise bekle ve tekrar dene
            if response.get('error') and '429' in str(response.get('error', '')):
                wait = 30 * (attempt + 1)  # 30, 60, 90 sn
                print(f"\n  ⚠️ Rate limit! {wait} sn bekleniyor (deneme {attempt+1}/{max_retries})")
                rate_limit_waits += 1
                time.sleep(wait)
                continue

            # Başarılı veya farklı hata - döngüden çık
            break

        except Exception as e:
            response = {"error": str(e)}
            break

    if response and response.get('error') is None:
        all_results[idx]['model_reasoning'] = response.get('reasoning')
        all_results[idx]['model_answer'] = response.get('answer')
        all_results[idx]['model_confidence'] = response.get('confidence')
        all_results[idx]['error'] = None
        fixed += 1
    else:
        still_error += 1

    # Her 20 sonuçta kaydet
    if (fixed + still_error) % 20 == 0:
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

# Final kayıt
with open(checkpoint_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# Güncel durum
gemini_data = [r for r in all_results if r['model'] == 'gemini-2.5-flash']
gemini_success = sum(1 for r in gemini_data if r.get('error') is None)
gemini_fail = sum(1 for r in gemini_data if r.get('error') is not None)

print(f"\n✓ Tamamlandı: {fixed} düzeltildi, {still_error} hâlâ hatalı")
print(f"  Rate limit bekleme sayısı: {rate_limit_waits}")
print(f"  Gemini toplam: {gemini_success} başarılı, {gemini_fail} hatalı")

Gemini hatalı sonuç: 249
► Gemini tekrar çalıştırılıyor...
  Strateji: 6 sn bekleme + 50'de bir 60 sn mola + rate limit retry
  Tahmini süre: ~29 dakika



Gemini retry:   0%|          | 0/249 [00:00<?, ?it/s]

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 4844.82ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1008.83ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3603.93ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2721.73ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1514.18ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3376.40ms



  ⚠️ Rate limit! 30 sn bekleniyor (deneme 1/3)


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1536.75ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1384.06ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1638.00ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3630.64ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3198.73ms


KeyboardInterrupt: 

In [ ]:
# ============================================================
# BÖLÜM 4.6: GEMİNİ EKSİKLERİNİ TAMAMLA (Yavaş mod)
# ============================================================
import time
from tqdm.notebook import tqdm

checkpoint_file = f"{CHECKPOINT_PATH}variant_generation_all_models.json"

with open(checkpoint_file, 'r', encoding='utf-8') as f:
    all_results = json.load(f)

# Sadece Gemini hatalılarını bul
gemini_errors = [i for i, r in enumerate(all_results)
                 if r['model'] == 'gemini-2.5-flash' and r.get('error') is not None]

print(f"Gemini hatalı sonuç: {len(gemini_errors)}")
print("► Gemini tekrar çalıştırılıyor (4 sn beklemeyle)...")
print(f"  Tahmini süre: ~{len(gemini_errors) * 5 / 60:.0f} dakika")

fixed = 0
still_error = 0

for idx in tqdm(gemini_errors, desc="Gemini retry"):
    r = all_results[idx]
    variant_id = r['variant_id']

    variant_row = df_variants[df_variants['variant_id'] == variant_id]
    if len(variant_row) == 0:
        still_error += 1
        continue
    variant_row = variant_row.iloc[0]

    time.sleep(4)

    try:
        response = generate_single_variant(
            context=variant_row['context'],
            question=variant_row['question'],
            model_name='gemini-2.5-flash'
        )

        if response.get('error') is None:
            all_results[idx]['model_reasoning'] = response.get('reasoning')
            all_results[idx]['model_answer'] = response.get('answer')
            all_results[idx]['model_confidence'] = response.get('confidence')
            all_results[idx]['error'] = None
            fixed += 1
        else:
            still_error += 1
    except Exception as e:
        still_error += 1

    if (fixed + still_error) % 25 == 0:
        with open(checkpoint_file, 'w', encoding='utf-8') as f:
            json.dump(all_results, f, ensure_ascii=False, indent=2)

# Final kayıt
with open(checkpoint_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

# Güncel durum
gemini_data = [r for r in all_results if r['model'] == 'gemini-2.5-flash']
gemini_success = sum(1 for r in gemini_data if r.get('error') is None)
gemini_fail = sum(1 for r in gemini_data if r.get('error') is not None)

print(f"\n✓ Tamamlandı: {fixed} düzeltildi, {still_error} hâlâ hatalı")
print(f"  Gemini toplam: {gemini_success} başarılı, {gemini_fail} hatalı")

Gemini hatalı sonuç: 249
► Gemini tekrar çalıştırılıyor (4 sn beklemeyle)...
  Tahmini süre: ~21 dakika


Gemini retry:   0%|          | 0/249 [00:00<?, ?it/s]


✓ Tamamlandı: 0 düzeltildi, 249 hâlâ hatalı
  Gemini toplam: 8 başarılı, 249 hatalı


In [ ]:
# Gemini hatalarının detayına bakalım
checkpoint_file = f"{CHECKPOINT_PATH}variant_generation_all_models.json"
with open(checkpoint_file, 'r', encoding='utf-8') as f:
    all_results = json.load(f)

gemini_errors = [r for r in all_results if r['model'] == 'gemini-2.5-flash' and r.get('error')]

# İlk 3 hatanın tam metnini göster
for r in gemini_errors[:3]:
    print(f"{r['variant_id']}: {r['error'][:300]}")
    print()

TR-01-017-v4: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-li

TR-1-018-v1: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-li

TR-1-018-v2: 429 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint: You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-li



#kontrol

##analiz

In [ ]:
# ============================================================
# VARYANT ANALİZİ - BÖLÜM 5: EVALUATION
# ============================================================
from tqdm.notebook import tqdm
from scipy.spatial.distance import cosine

# Checkpoint'ten generation sonuçlarını yükle
checkpoint_file = f"{CHECKPOINT_PATH}variant_generation_all_models.json"
with open(checkpoint_file, 'r', encoding='utf-8') as f:
    all_results = json.load(f)

df_gen = pd.DataFrame(all_results)

# Sadece başarılı sonuçları al ve şimdilik Gemini'yi çıkar
df_valid = df_gen[
    (df_gen['error'].isna()) &
    (df_gen['model_reasoning'].notna()) &
    (df_gen['model_reasoning'].apply(lambda x: len(str(x)) > 0)) &
    (df_gen['model'] != 'gemini-2.5-flash')
].copy().reset_index(drop=True)

print(f"Toplam generation: {len(df_gen)}")
print(f"Geçerli (Gemini hariç): {len(df_valid)}")
print(f"\nModel dağılımı:")
for model in df_valid['model'].unique():
    print(f"  {model}: {len(df_valid[df_valid['model']==model])}")

# Varyant verisinden gold_evidence_list'i eşleştir
variant_evidence = df_variants[['variant_id', 'gold_evidence_list']].copy()
df_valid = df_valid.merge(variant_evidence, on='variant_id', how='left')

# Faithfulness hesapla
print(f"\n► Evaluation başlıyor ({len(df_valid)} sonuç)...")
print(f"  Tahmini süre: ~{len(df_valid) * 0.5 / 60:.0f} dakika")

evaluation_results = []
errors = []

for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Evaluating"):
    try:
        reasoning = str(row['model_reasoning'])[:5000]
        evidence_list = row['gold_evidence_list']

        semantic_scores = []
        nli_scores = []

        for evidence in evidence_list:
            evidence = str(evidence).strip()[:2000]
            if not evidence:
                continue

            # Semantic similarity
            try:
                embeddings = emb_model.encode([reasoning, evidence])
                sem_sim = 1 - cosine(embeddings[0], embeddings[1])
                sem_sim = max(0.0, min(1.0, sem_sim))
            except:
                sem_sim = 0.5
            semantic_scores.append(sem_sim)

            # NLI score
            try:
                result = nli_pipe(
                    reasoning,
                    candidate_labels=["entailment", "neutral", "contradiction"],
                    hypothesis_template="{}"
                )
                scores = dict(zip(result["labels"], result["scores"]))
                nli_score = (scores.get("entailment", 0) - scores.get("contradiction", 0) + 1) / 2
            except:
                nli_score = 0.5
            nli_scores.append(nli_score)

        avg_semantic = np.mean(semantic_scores) if semantic_scores else 0.5
        avg_nli = np.mean(nli_scores) if nli_scores else 0.5
        faithfulness = 0.4 * avg_semantic + 0.6 * avg_nli

        # Accuracy
        model_answer = str(row['model_answer']).strip().lower() if row['model_answer'] else ""
        gold_answer = str(row.get('gold_answer', '')).strip().lower() if row.get('gold_answer') else ""

        # gold_answer'ı varyant verisinden al
        variant_row = df_variants[df_variants['variant_id'] == row['variant_id']]
        if len(variant_row) > 0:
            gold_answer = str(variant_row.iloc[0]['gold_answer']).strip().lower()

        exact_match = model_answer == gold_answer

        # Confidence
        confidence = row['model_confidence'] if row['model_confidence'] else 50
        conf_norm = confidence / 100

        # Epistemic gap
        epistemic_gap = abs(conf_norm - faithfulness)

        # Hallucination risk
        if conf_norm > 0.7 and faithfulness < 0.5:
            hall_risk = "YÜKSEK"
        elif conf_norm > 0.7 and faithfulness < 0.7:
            hall_risk = "ORTA"
        else:
            hall_risk = "DÜŞÜK"

        # Calibration
        if epistemic_gap < 0.15:
            cal_type = "İyi Kalibre"
        elif conf_norm > faithfulness:
            cal_type = "Overconfident"
        else:
            cal_type = "Underconfident"

        evaluation_results.append({
            'variant_id': row['variant_id'],
            'base_question_id': row['base_question_id'],
            'category': row['category'],
            'difficulty': row['difficulty'],
            'variant_type': row['variant_type'],
            'model': row['model'],
            'semantic_similarity': round(avg_semantic, 4),
            'nli_score': round(avg_nli, 4),
            'faithfulness_score': round(faithfulness, 4),
            'exact_match': exact_match,
            'accuracy_score': 1 if exact_match else 0,
            'confidence_normalized': round(conf_norm, 4),
            'epistemic_gap': round(epistemic_gap, 4),
            'hallucination_risk': hall_risk,
            'calibration_type': cal_type,
            'model_answer': row['model_answer'],
            'gold_answer': gold_answer
        })

    except Exception as e:
        errors.append({'variant_id': row['variant_id'], 'model': row['model'], 'error': str(e)})

df_evaluated = pd.DataFrame(evaluation_results)

# Kaydet
eval_path = f"{RESULTS_PATH}variant_evaluation_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_evaluated.to_csv(eval_path, index=False, encoding='utf-8-sig')

print(f"\n✓ Evaluation tamamlandı: {len(df_evaluated)} sonuç")
print(f"  Hatalı: {len(errors)}")
print(f"  Kaydedildi: {eval_path}")

print("\n=== MODEL BAZLI ÖZET ===")
for model in df_evaluated['model'].unique():
    m = df_evaluated[df_evaluated['model'] == model]
    print(f"\n  {model}:")
    print(f"    Faithfulness: {m['faithfulness_score'].mean():.3f}")
    print(f"    Accuracy: {m['accuracy_score'].mean()*100:.1f}%")
    print(f"    Confidence: {m['confidence_normalized'].mean()*100:.1f}%")
    print(f"    Epistemic Gap: {m['epistemic_gap'].mean():.3f}")

Toplam generation: 1028
Geçerli (Gemini hariç): 762

Model dağılımı:
  gpt-4o-mini: 256
  gpt-4o: 257
  claude-sonnet-4-20250514: 249

► Evaluation başlıyor (762 sonuç)...
  Tahmini süre: ~6 dakika


Evaluating:   0%|          | 0/762 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✓ Evaluation tamamlandı: 762 sonuç
  Hatalı: 0
  Kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/variant_evaluation_20260414_1440.csv

=== MODEL BAZLI ÖZET ===

  gpt-4o-mini:
    Faithfulness: 0.670
    Accuracy: 39.5%
    Confidence: 93.8%
    Epistemic Gap: 0.268

  gpt-4o:
    Faithfulness: 0.641
    Accuracy: 39.7%
    Confidence: 98.6%
    Epistemic Gap: 0.345

  claude-sonnet-4-20250514:
    Faithfulness: 0.650
    Accuracy: 25.3%
    Confidence: 95.7%
    Epistemic Gap: 0.310


In [ ]:
# ============================================================
# VARYANT ANALİZİ - BÖLÜM 5.5: SOFT ACCURACY HESAPLAMASI
# Var olan evaluation CSV'sine soft_accuracy_score kolonu ekler
# ============================================================
from tqdm.notebook import tqdm

# Var olan evaluation CSV'sini yükle
eval_csv = f"{RESULTS_PATH}variant_evaluation_20260414_1440.csv"
df_eval = pd.read_csv(eval_csv)
print(f"✓ Evaluation CSV yüklendi: {len(df_eval)} satır")
print(f"  Mevcut kolonlar: {len(df_eval.columns)}")

# faithfulness_utils'in modelleri kullanabilmesi için set_models çağır
set_models(emb_model, nli_pipe, None, None, None)
print("✓ Modeller faithfulness_utils'e bağlandı")

# Sanity test - calculate_soft_accuracy gerçekten çalışıyor mu
print("\n=== Sanity test ===")
test = calculate_soft_accuracy("400", "400")
print(f"  Birebir test: {test}")
assert test['accuracy_score'] == 1, "Sanity test başarısız - fonksiyon doğru çalışmıyor"
print("  ✓ Soft accuracy fonksiyonu doğru çalışıyor")

# 762 satıra soft accuracy hesapla
print(f"\n► {len(df_eval)} satıra soft accuracy hesaplanıyor...")
print(f"  Tahmini süre: ~{len(df_eval) * 0.4 / 60:.0f} dakika")

soft_results = []
for idx, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc="Soft Acc"):
    try:
        result = calculate_soft_accuracy(row['model_answer'], row['gold_answer'])
        soft_results.append({
            'soft_match': bool(result['soft_match']),
            'soft_accuracy_score': int(result['accuracy_score']),
            'match_type': str(result['match_type']),
            'answer_semantic_sim': float(result['semantic_sim']),
            'answer_nli_entailment': float(result['nli_entailment']),
        })
    except Exception as e:
        soft_results.append({
            'soft_match': False,
            'soft_accuracy_score': 0,
            'match_type': f'error: {str(e)[:100]}',
            'answer_semantic_sim': 0.0,
            'answer_nli_entailment': 0.0,
        })

# Soft accuracy kolonlarını df_eval'a ekle
soft_df = pd.DataFrame(soft_results)
df_eval_with_soft = pd.concat([df_eval.reset_index(drop=True), soft_df], axis=1)

# Yeni CSV olarak kaydet (orijinale dokunma)
new_path = f"{RESULTS_PATH}variant_evaluation_with_soft_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_eval_with_soft.to_csv(new_path, index=False, encoding='utf-8-sig')

print(f"\n✓ Soft accuracy hesaplandı ve kaydedildi:")
print(f"  Dosya: {new_path}")
print(f"  Toplam: {len(df_eval_with_soft)} satır, {len(df_eval_with_soft.columns)} kolon")

# Model bazlı özet
print("\n=== MODEL BAZLI SOFT ACCURACY ÖZETİ ===")
for model in df_eval_with_soft['model'].unique():
    m = df_eval_with_soft[df_eval_with_soft['model'] == model]
    exact_rate = m['accuracy_score'].mean() * 100
    soft_rate = m['soft_accuracy_score'].mean() * 100
    print(f"  {model:<32}: N={len(m)}, Exact={exact_rate:.1f}%, Soft={soft_rate:.1f}%")

✓ Evaluation CSV yüklendi: 762 satır
  Mevcut kolonlar: 17
✓ Modeller faithfulness_utils'e bağlandı

=== Sanity test ===
  Birebir test: {'exact_match': True, 'soft_match': True, 'accuracy_score': 1, 'match_type': 'exact', 'semantic_sim': 1.0, 'nli_entailment': 1.0}
  ✓ Soft accuracy fonksiyonu doğru çalışıyor

► 762 satıra soft accuracy hesaplanıyor...
  Tahmini süre: ~5 dakika


Soft Acc:   0%|          | 0/762 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



✓ Soft accuracy hesaplandı ve kaydedildi:
  Dosya: /content/drive/MyDrive/BitirmeProjesi/results/variant_evaluation_with_soft_20260518_2108.csv
  Toplam: 762 satır, 22 kolon

=== MODEL BAZLI SOFT ACCURACY ÖZETİ ===
  gpt-4o-mini                     : N=256, Exact=39.5%, Soft=54.3%
  gpt-4o                          : N=257, Exact=39.7%, Soft=53.3%
  claude-sonnet-4-20250514        : N=249, Exact=25.3%, Soft=39.4%


In [ ]:
# VARYANT ANALİZİ - BÖLÜM 6: BASE vs VARYANT KARŞILAŞTIRMA

print("► Base-Varyant karşılaştırması yapılıyor...\n")

# Her model için base faithfulness skorlarını hazırla
# Base eval dosyalarındaki sütun isimlerini kontrol edelim önce
sample_base = pd.read_csv(list(BASE_EVAL_FILES.values())[0], nrows=2)
print(f"Base eval sütunları: {list(sample_base.columns)}\n")

comparison_rows = []

for model_name in ['gpt-4o-mini', 'gpt-4o', 'claude-sonnet-4-20250514']:

    # Base evaluation yükle
    df_base = pd.read_csv(BASE_EVAL_FILES[model_name])

    # Varyant evaluation'dan bu modelin sonuçlarını al
    df_var_model = df_evaluated[df_evaluated['model'] == model_name].copy()

    for _, var_row in df_var_model.iterrows():
        base_id = str(var_row['base_question_id']).strip()

        # Base sorunun faithfulness skorunu bul
        base_match = df_base[df_base['question_id'].str.strip() == base_id]

        if len(base_match) == 0:
            continue

        base_row = base_match.iloc[0]

        base_faith = base_row['faithfulness_score']
        var_faith = var_row['faithfulness_score']
        faith_diff = var_faith - base_faith  # pozitif = varyant daha iyi, negatif = düşüş

        base_acc = base_row.get('soft_accuracy_score', base_row.get('accuracy_score', 0))
        var_acc = var_row['accuracy_score']

        comparison_rows.append({
            'variant_id': var_row['variant_id'],
            'base_question_id': base_id,
            'category': var_row['category'],
            'difficulty': var_row['difficulty'],
            'variant_type': var_row['variant_type'],
            'model': model_name,
            'base_faithfulness': round(base_faith, 4),
            'variant_faithfulness': round(var_faith, 4),
            'faithfulness_diff': round(faith_diff, 4),
            'base_accuracy': base_acc,
            'variant_accuracy': var_acc,
            'accuracy_changed': base_acc != var_acc,
            'base_confidence': round(base_row.get('confidence_normalized', 0.5), 4),
            'variant_confidence': round(var_row['confidence_normalized'], 4),
            'base_epistemic_gap': round(base_row.get('epistemic_gap', 0), 4),
            'variant_epistemic_gap': round(var_row['epistemic_gap'], 4)
        })

df_comparison = pd.DataFrame(comparison_rows)

# Kaydet
comp_path = f"{RESULTS_PATH}variant_comparison_{datetime.now().strftime('%Y%m%d_%H%M')}.csv"
df_comparison.to_csv(comp_path, index=False, encoding='utf-8-sig')

print(f"✓ Karşılaştırma tamamlandı: {len(df_comparison)} base-varyant çifti")
print(f"  Kaydedildi: {comp_path}\n")

# GENEL ÖZET
print("=" * 70)
print("VARYANT ANALİZİ - GENEL ÖZET")
print("=" * 70)

print(f"\n► Toplam karşılaştırma: {len(df_comparison)}")
print(f"  Ortalama faithfulness farkı: {df_comparison['faithfulness_diff'].mean():.4f}")
print(f"  Faithfulness düşen varyant oranı: {(df_comparison['faithfulness_diff'] < 0).mean()*100:.1f}%")
print(f"  Faithfulness artan varyant oranı: {(df_comparison['faithfulness_diff'] > 0).mean()*100:.1f}%")
print(f"  Accuracy değişen varyant oranı: {df_comparison['accuracy_changed'].mean()*100:.1f}%")

# MODEL BAZLI KARŞILAŞTIRMA

print(f"\n{'='*70}")
print("MODEL BAZLI KARŞILAŞTIRMA")
print(f"{'='*70}")

for model in ['gpt-4o-mini', 'gpt-4o', 'claude-sonnet-4-20250514']:
    m = df_comparison[df_comparison['model'] == model]
    print(f"\n  {model}:")
    print(f"    Base ort. faithfulness:    {m['base_faithfulness'].mean():.3f}")
    print(f"    Varyant ort. faithfulness: {m['variant_faithfulness'].mean():.3f}")
    print(f"    Ortalama fark:             {m['faithfulness_diff'].mean():.4f}")
    print(f"    Faithfulness düşen:        {(m['faithfulness_diff'] < 0).mean()*100:.1f}%")
    print(f"    Accuracy değişen:          {m['accuracy_changed'].mean()*100:.1f}%")


# VARYANT TİPİ BAZLI ETKİ ANALİZİ

print(f"\n{'='*70}")
print("VARYANT TİPİ BAZLI ETKİ ANALİZİ")
print(f"{'='*70}")

vt_analysis = df_comparison.groupby('variant_type').agg(
    count=('faithfulness_diff', 'size'),
    mean_diff=('faithfulness_diff', 'mean'),
    std_diff=('faithfulness_diff', 'std'),
    pct_decreased=('faithfulness_diff', lambda x: (x < 0).mean() * 100)
).round(4).sort_values('mean_diff')

print(f"\n{'Varyant Tipi':<30} {'N':>4} {'Ort.Fark':>10} {'Std':>8} {'Düşen%':>8}")
print("-" * 65)
for vt, row in vt_analysis.iterrows():
    print(f"  {vt:<28} {int(row['count']):>4} {row['mean_diff']:>+10.4f} {row['std_diff']:>8.4f} {row['pct_decreased']:>7.1f}%")

# KATEGORİ BAZLI ETKİ ANALİZİ

print(f"\n{'='*70}")
print("KATEGORİ BAZLI ETKİ ANALİZİ")
print(f"{'='*70}")

cat_analysis = df_comparison.groupby('category').agg(
    count=('faithfulness_diff', 'size'),
    mean_diff=('faithfulness_diff', 'mean'),
    std_diff=('faithfulness_diff', 'std'),
    pct_decreased=('faithfulness_diff', lambda x: (x < 0).mean() * 100)
).round(4).sort_values('mean_diff')

print(f"\n{'Kategori':<30} {'N':>4} {'Ort.Fark':>10} {'Std':>8} {'Düşen%':>8}")
print("-" * 65)
for cat, row in cat_analysis.iterrows():
    print(f"  {cat:<28} {int(row['count']):>4} {row['mean_diff']:>+10.4f} {row['std_diff']:>8.4f} {row['pct_decreased']:>7.1f}%")


# MODEL × VARYANT TİPİ ÇAPRAZ TABLO
print(f"\n{'='*70}")
print("MODEL × VARYANT TİPİ ÇAPRAZ TABLO (Ort. Faithfulness Farkı)")
print(f"{'='*70}\n")

cross = df_comparison.pivot_table(
    values='faithfulness_diff',
    index='variant_type',
    columns='model',
    aggfunc='mean'
).round(4)

print(cross.to_string())

print(f"\n\n✓ Analiz tamamlandı!")

► Base-Varyant karşılaştırması yapılıyor...

Base eval sütunları: ['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count', 'model_reasoning', 'model_answer', 'model_confidence', 'error', 'semantic_similarity', 'nli_score', 'faithfulness_score', 'evidence_recall', 'exact_match', 'accuracy_score', 'confidence_normalized', 'epistemic_gap', 'hallucination_risk', 'calibration_type', 'reasoning_length', 'evidence_length', 'length_ratio', 'soft_match', 'soft_accuracy_score', 'match_type', 'answer_semantic_sim', 'answer_nli_entailment']

✓ Karşılaştırma tamamlandı: 762 base-varyant çifti
  Kaydedildi: /content/drive/MyDrive/BitirmeProjesi/results/variant_comparison_20260414_1445.csv

VARYANT ANALİZİ - GENEL ÖZET

► Toplam karşılaştırma: 762
  Ortalama faithfulness farkı: -0.0036
  Faithfulness düşen varyant oranı: 53.0%
  Faithfulness artan varyant oranı: 46.9%
  Accuracy değişen varyant oranı: 41.1%

MODEL BAZLI K